# **Mapping short-let pressure in Bristol: A neighbourhood clustering analysis of Airbnb listings**

## Objectives

* This project analyses Inside Airbnb listing data for Bristol to help local authority policy and planning teams understand where short-term letting is concentrated and what form it takes. 

Using unsupervised machine learning(ADD DETAIL TBC), it groups Bristol’s neighbourhoods into distinct short-let profiles based on measures such as listing density, entire-home share, host portfolio size and annual availability. An interactive dashboard lets users explore each profile and compare neighbourhoods. 

This is an independent portfolio project written for an imagined council audience; it is not commissioned by or affiliated with any local authority. 

## Project Plan

This project follows a structured analytics workflow. Each stage is documented in the notebook with the reasoning behind decisions, not just the code.

1. **Define the problem.** Set the business question, target audience (an imagined local authority policy team) and how "short-let pressure" will be measured.
2. **Extract.** Load the Inside Airbnb Bristol snapshot (listings, neighbourhood lookup and boundary GeoJSON), keeping raw files unmodified.
3. **Check the reference data.** Confirm that neighbourhood names are consistent across all three files so they can be joined safely.
4. **Investigate the listings data.** Inspect structure, data types, missing values, constant columns and distributions.
5. **Clean and transform.** Drop empty, constant and personal-data columns (each with a recorded reason), convert data types, handle missing values and check outliers.
6. **Build the neighbourhood-level table.** Aggregate listings into one row per neighbourhood using the chosen pressure measures, and engineer new features.
7. **Exploratory data analysis.** Run descriptive statistics, distributions, correlations and initial visualisations, and test at least one hypothesis.
8. **Machine learning.** Compare candidate clustering approaches, justify the choice, prepare and scale features, select the number of groups and evaluate the result.
9. **Interpret the results.** Profile each neighbourhood group, visualise the core findings and state their limitations.
10. **Build an interactive dashboard.** Let users explore neighbourhood groups through filters and maps, in [chosen tool].
11. **Document and reflect.** Complete the README, record limitations and alternative approaches, and reflect on what was learned.

## Inputs

* Data: Inside Airbnb, snapshot dated 22nd September 2026, licensed: https://insideairbnb.com/get-the-data/

The files used:  

- `listings.csv.gz` - Detailed listings, the main table for measuring short-let activity 
- `neighbourhoods.csv` - Neighbourhood names, the unit of analysis 
- `neighbourhoods.geojson` - Neighbourhood boundaries, for map visualisations later 

## Outputs

* Write here which files, code or artefacts you generate by the end of the notebook 

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\chheb\\OneDrive\\Documents\\Programming\\Mapping_Short_Let-Pressure_in_Bristol\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\chheb\\OneDrive\\Documents\\Programming\\Mapping_Short_Let-Pressure_in_Bristol'

# Defining the problem 

# Importing the data 

### Method
- Raw files are stored unmodified in `data/raw/`. All cleaning happens on copies, so the original download can always be recovered.
- `listings.csv.gz` is read directly by pandas, which decompresses it automatically.
- The GeoJSON is loaded with Python's `json` module as a dictionary. [Why this rather than geopandas?]
- Paths are written [relative to the notebook / as raw strings] so the notebook runs on other machines. 



In [4]:
from pathlib import Path
import pandas as pd
import json

In [5]:
DATA_RAW = Path(r"c:\Users\chheb\OneDrive\Documents\Programming\Mapping_Short_Let-Pressure_in_Bristol\Data\raw")  


listings = pd.read_csv(DATA_RAW / "listings.csv.gz")        
neighbourhoods = pd.read_csv(DATA_RAW / "neighbourhoods.csv")

with open(DATA_RAW / "neighbourhoods.geojson", encoding="utf-8") as f:
    neighbourhoods_geo = json.load(f)  

### Confirming the import worked
- The data folder path was checked with `DATA_RAW.exists()`, which returned `True`.
- `listings.shape` returned **(2,932, 90)**: 2,932 listings and 90 columns.


In [6]:
print(DATA_RAW.exists())


True


In [7]:
print(listings.shape)

(2932, 90)


In [8]:
listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,70820,https://www.airbnb.com/rooms/70820,20260922161704,2026-09-25,NaN,City View - Sarah's double room.,A good sized room with a comfy double bed and ...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,360195,...,4.93,4.76,4.76,NaN,NaN,5,0,5,0,1.36
1,176306,https://www.airbnb.com/rooms/176306,20260922161704,2026-09-25,NaN,The White Room - Central Bristol Art House Ga...,"A small double room in an artist's house, with...",NaN,https://a0.muscache.com/pictures/e1e7fdfc-9db5...,708175,...,4.90,4.78,4.69,NaN,NaN,1,0,1,0,0.36
2,307634,https://www.airbnb.com/rooms/307634,20260922161704,2026-09-25,NaN,"'The Sunset Room' Peaceful, Clear Home","Close to Bristol and Bath Railway Path, leadin...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,1582810,...,4.84,4.86,4.74,NaN,NaN,3,0,3,0,0.47
3,390781,https://www.airbnb.com/rooms/390781,20260922161704,2026-09-25,previous scrape,"Cosy large room nr TV studios, Paintworks & st...",A room with a shared bathroom in an old Victor...,NaN,https://a0.muscache.com/pictures/9755283/2e0bb...,1954565,...,4.93,4.72,4.84,NaN,NaN,2,0,2,0,0.34
4,407485,https://www.airbnb.com/rooms/407485,20260922161704,2026-09-25,NaN,Bristol Beautiful Garden Apartment,"<b>The space</b><br />This is a sunny, light h...",NaN,https://a0.muscache.com/pictures/58023936/aa2b...,2029719,...,4.92,4.79,4.70,NaN,NaN,1,1,0,0,2.35


### First observations
- 2,932 listings is a modest sample for the whole city, so [what might this mean for clustering by neighbourhood?].
- 90 columns is more than the analysis needs. A later step will select the relevant ones and justify each choice.
- [Anything else you noticed from `info()`, `head()` or the missing-value check.]

---

# Exploratory data analysis

## 1. Lookup files


`neighbourhoods.csv` and `neighbourhoods.geojson` are lookup files, not analytical data. They supply the area names (and, for the GeoJSON, the boundary shapes) that listings are grouped by. The aim is to confirm they can be joined to the listings safely, not to explore distributions.

**Steps**
1. Checked the CSV's structure with `.shape`, `.info()` and `.head()`.
2. Counted nulls per column and duplicates in the name column.
3. Checked for stray whitespace in names, since an invisible trailing space would make two identical-looking names fail to match.
4. Compared the area names in `listings["neighbourhood_cleansed"]` against the lookup table using set differences, in both directions.
5. Inspected the GeoJSON structure (top-level keys, number of features, and the properties of the first feature) to find which key holds the area name.
6. Compared the GeoJSON area names against the lookup table, in both directions.


In [9]:
neighbourhoods.shape

(34, 2)

In [10]:
neighbourhoods.info()

<class 'pandas.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   neighbourhood_group  0 non-null      float64
 1   neighbourhood        34 non-null     str    
dtypes: float64(1), str(1)
memory usage: 1.1 KB


In [11]:
neighbourhoods.head()

,neighbourhood_group,neighbourhood
0,NaN,Ashley
1,NaN,Avonmouth & Lawrence Weston
2,NaN,Bedminster
3,NaN,Bishopston & Ashley Down
4,NaN,Bishopsworth


In [12]:
print(neighbourhoods.isnull().sum())
print("Duplicated names:", neighbourhoods["neighbourhood"].duplicated().sum())

neighbourhood_group    34
neighbourhood           0
dtype: int64
Duplicated names: 0


In [13]:
names = neighbourhoods["neighbourhood"]
print("Names with stray whitespace:", (names != names.str.strip()).sum())

Names with stray whitespace: 0


In [14]:
print("Null areas in listings:", listings["neighbourhood_cleansed"].isnull().sum())

listing_areas = set(listings["neighbourhood_cleansed"].dropna().unique())
lookup_areas = set(neighbourhoods["neighbourhood"])

print("In listings but not in lookup:", listing_areas - lookup_areas)
print("In lookup but not in listings:", lookup_areas - listing_areas)

Null areas in listings: 0
In listings but not in lookup: set()
In lookup but not in listings: set()


**Findings: CSV lookup**
- The table has **34 areas** and 2 columns. `.head()` shows area names (for example Ashley, Avonmouth & Lawrence Weston, Bedminster) alongside a blank `neighbourhood_group`.
- There are no duplicated names and no whitespace issues.
- `neighbourhood_group` is **empty for all 34 rows**, which is why pandas read it as `float64`. It holds no information for Bristol.

**Findings: listings against lookup**
- All 2,932 listings have an area (0 nulls).
- The set of areas in `listings` is **identical** to the set in the lookup table. No listing is orphaned and no area is empty.

In [15]:
print(neighbourhoods_geo.keys())
print("Number of features:", len(neighbourhoods_geo["features"]))
print(neighbourhoods_geo["features"][0]["properties"]) 

dict_keys(['type', 'features'])
Number of features: 34
{'neighbourhood': 'Windmill Hill', 'neighbourhood_group': None}


In [16]:
geo_areas = {f["properties"]["neighbourhood"] for f in neighbourhoods_geo["features"]}

print("In GeoJSON but not in lookup:", geo_areas - lookup_areas)
print("In lookup but not in GeoJSON:", lookup_areas - geo_areas)

In GeoJSON but not in lookup: set()
In lookup but not in GeoJSON: set()


**Findings: GeoJSON**
- Top-level keys are `type` and `features`, with **34 features**, one per area. This matches the CSV row count.
- Each feature's `properties` holds the area name under the key `neighbourhood` (plus an empty `neighbourhood_group`, consistent with the CSV).
- GeoJSON names match the lookup table **exactly in both directions**: no names appear in one but not the other.

**Decisions**
- **Drop `neighbourhood_group`.** It contains no values, so it cannot support any analysis.
- **No renaming or text cleaning needed.** Names match exactly across all three files, so I can join on `neighbourhood_cleansed` and use the GeoJSON for maps without a mapping step.
- **Use `neighbourhood_cleansed` as the area key.** The raw `neighbourhood` column in the listings file is entirely empty.
- **Leave the boundary shapes untouched for now.** They are only needed for the dashboard map.

**Limitations and open questions**
- These checks show that names are consistent across files, not that the boundaries suit this analysis. [Add what the Inside Airbnb page says about where the areas come from.] Any external data I join later (for example, household counts) will need to use the same boundaries.
- Areas are very uneven in size, from [smallest] to [largest] listings. This affects how reliable any area-level percentage will be, and is addressed when the neighbourhood-level table is built.

## 2. 'listings': analytical data

### A. Structure and data types

In [17]:
listings.describe()

,id,scrape_id,neighborhood_overview,host_id,host_profile_id,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,2.932000e+03,2.932000e+03,0.0,2.932000e+03,2.932000e+03,0.0,2932.000000,2932.000000,2932.000000,2932.000000,...,2644.000000,2644.000000,2644.000000,0.0,0.0,2932.000000,2932.00000,2932.000000,2932.000000,2644.000000
mean,9.072239e+17,2.026092e+13,NaN,3.503762e+16,1.481828e+18,NaN,7.926330,5.256139,5.155866,5.189632,...,4.851880,4.767613,4.691566,NaN,NaN,12.156207,11.28206,0.827763,0.011937,1.692663
std,6.413870e+17,0.000000e+00,NaN,2.424918e+17,5.380515e+16,NaN,3.822936,3.316709,3.671786,3.316705,...,0.312867,0.283084,0.393268,NaN,NaN,28.026407,27.64082,1.520924,0.244057,1.886890
min,7.082000e+04,2.026092e+13,NaN,2.381000e+03,1.462506e+18,NaN,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,NaN,NaN,1.000000,0.00000,0.000000,0.000000,0.010000
25%,5.134202e+07,2.026092e+13,NaN,3.759181e+07,1.463080e+18,NaN,4.000000,2.750000,2.000000,2.000000,...,4.830000,4.680000,4.600000,NaN,NaN,1.000000,1.00000,0.000000,0.000000,0.400000
50%,1.055818e+18,2.026092e+13,NaN,1.226961e+08,1.467231e+18,NaN,9.000000,5.000000,4.000000,5.000000,...,4.950000,4.840000,4.800000,NaN,NaN,2.000000,1.00000,0.000000,0.000000,1.010000
75%,1.479047e+18,2.026092e+13,NaN,4.267019e+08,1.469991e+18,NaN,11.000000,8.000000,9.000000,8.000000,...,5.000000,4.950000,4.910000,NaN,NaN,6.000000,4.00000,1.000000,0.000000,2.380000
max,1.779673e+18,2.026092e+13,NaN,1.775395e+18,1.775395e+18,NaN,18.000000,11.000000,15.000000,11.000000,...,5.000000,5.000000,5.000000,NaN,NaN,129.000000,126.00000,11.000000,5.000000,14.520000


In [18]:
listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 2932 entries, 0 to 2931
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            2932 non-null   int64  
 1   listing_url                                   2932 non-null   str    
 2   scrape_id                                     2932 non-null   int64  
 3   last_scraped                                  2932 non-null   str    
 4   source                                        705 non-null    str    
 5   name                                          2932 non-null   str    
 6   description                                   2924 non-null   str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   2932 non-null   str    
 9   host_id                                       2932 non-null   int64  
 10 

In [22]:
listings.dtypes.value_counts()
listings.select_dtypes("object").columns.tolist() 

C:\Users\chheb\AppData\Local\Temp\ipykernel_4880\2973873790.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  listings.select_dtypes("object").columns.tolist()


['listing_url',
 'last_scraped',
 'source',
 'name',
 'description',
 'picture_url',
 'host_url',
 'host_profile_url',
 'host_name',
 'host_location',
 'host_about',
 'host_is_superhost',
 'host_picture_url',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood_cleansed',
 'property_type',
 'room_type',
 'bathrooms_text',
 'amenities',
 'price',
 'price_quote_checkin_date',
 'price_quote_checkout_date',
 'price_quote_raw',
 'has_availability',
 'calendar_last_scraped',
 'first_review',
 'last_review']

In [23]:
listings[["price", "first_review", "last_review"]].head(10)

,price,first_review,last_review
0,$52.33,2013-10-18,2026-07-29
1,$58.00,2012-11-18,2025-12-02
2,$53.50,2017-06-04,2026-09-19
3,NaN,2012-03-26,2026-04-23
4,$134.00,2012-04-30,2026-09-06
5,$77.00,2012-07-23,2026-09-17
6,$272.00,2012-04-30,2026-09-13
7,$170.00,2012-11-15,2026-09-17
8,NaN,2018-03-29,2020-03-01
9,$189.00,2013-07-09,2026-09-13


### B. Missing values

### C. Empty columns

### D. Duplicates and identifiers

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [20]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)


IndentationError: expected an indented block after 'try' statement on line 2 (553063055.py, line 5)